# RL Group Project 3 — Shared setup (start every notebook from this file)

**What this is.** The code every part of the project shares: environments, seeds, frame preprocessing, the training log, and the 200-game evaluation. Pong Part 1 (`pong_full`), Pong Part 2 (`pong_5k`), Breakout (`breakout`) and the final report all use it, so every agent is scored the same way and the TA's re-run reproduces our numbers.

**How to use it.**
1. Copy this notebook and rename the copy for your part: `pong_full.ipynb`, `pong_5k.ipynb` or `breakout.ipynb`.
2. Leave the cells above the **"Your agent starts here"** line as they are. If something shared needs to change, change it here in the repo and tell the group, so all four notebooks stay identical.
3. Build your agent below that line.
4. At the end, D pastes this shared section **once** at the top of the final submission notebook, followed by each part's code. That gives the one `.ipynb` + HTML that Canvas asks for.

**What's in here**

| Section | What it gives you | Who uses it |
| --- | --- | --- |
| Constants | Folder names, 200 eval games, eval seeds, frameskip 4, no sticky actions | Everyone |
| `set_global_seeds(seed)` | Seeds Python, NumPy, TensorFlow; deterministic TF ops | Everyone |
| `make_env(game)` | `ALE/<game>-v5`, an action every 4 frames | Everyone |
| `preprocess_pong`, `PongFrameDiff`, `ball_visible_pong` | 80×80 Pong frames, frame differencing, "is the ball on screen?" | A, B |
| `preprocess_breakout`, `breakout_fire_needed` | 82×72 Breakout frames, auto-FIRE after reset / lost life | C |
| `FrameStack(preprocess_fn, k=4)` | Stacks the last k frames as (H, W, k), Keras channels-last like the class models, so the net sees motion | A, B, C |
| `TrainingLogger(agent, resume=True)` | `results/<agent>_train_log.csv` + `_meta.json`; resumes after a Colab disconnect | A, B, C |
| `fetch_weights(agent)` | Downloads `models/<agent>.weights.h5` from the GitHub repo if it isn't on disk (so the TA can run with `TRAIN = False`) | A, B, C, D |
| `evaluate(env_fn, policy, agent_name=...)` | The 200-game greedy evaluation → `results/<agent>_eval_200.csv` | A, B, C |
| `random_baseline(game)` | Random-play baseline → `results/<game>_random_eval_200.csv` | C, D |

**Your policy object** only needs `policy.act(obs) -> int` (`obs` is the **raw** env frame; do your preprocessing inside) and, optionally, `policy.reset()` to clear frame-diff / frame-stack state between games. For evaluation, `act` must be greedy: no epsilon, no sampling.

**Files every agent delivers to D**

| File | Contents |
| --- | --- |
| `results/<agent>_train_log.csv` | One row per training game: game, score, 100-game running mean, frames, wall-clock seconds |
| `results/<agent>_eval_200.csv` | 200 rows: game, score, steps |
| `results/<agent>_meta.json` | Hyperparameters, training games, training hours, hardware, seed |
| `models/<agent>.weights.h5` | Final trained weights from Keras `model.save_weights(...)` (commit to the repo; keep each under ~50 MB) |
| `notes/<agent>_writeup.md` | 5–10 bullets on what you tried / what worked / what failed, plus 2–4 key code chunks |

Agent names: `pong_full`, `pong_5k`, `breakout`. Pong-5k also delivers `results/pong_5k_checkpoint_evals.csv`.

Checkpoints during training go in `models/checkpoints/`. That folder is git-ignored because checkpoints with replay buffers get large.

## Install
Uncomment on Colab or on a fresh machine. The versions match `requirements.txt` in the repo.

In [ ]:
# %pip install -q "gymnasium[atari]==1.4.0" "ale-py==0.12.1" "numpy==2.5.5" "pandas==3.0.1" "tensorflow==2.21.0" "matplotlib==3.10.8"

In [ ]:
import json
import os
import platform
import random
import time
import urllib.request

import numpy as np
import pandas as pd
import gymnasium as gym
import ale_py
import tensorflow as tf

gym.register_envs(ale_py)  # registers the "ALE/..." environment ids with gymnasium

print(f"gymnasium {gym.__version__} | ale-py {ale_py.__version__} | numpy {np.__version__} "
      f"| pandas {pd.__version__} | tensorflow {tf.__version__}")

## Constants
Every setting the whole team must agree on lives here, in one place.

In [ ]:
RESULTS_DIR = "results"            # CSV / JSON outputs
MODELS_DIR = "models"              # final weights: models/<agent>.weights.h5
CHECKPOINT_DIR = os.path.join(MODELS_DIR, "checkpoints")  # mid-training saves (git-ignored)
NOTES_DIR = "notes"                # notes/<agent>_writeup.md

EVAL_GAMES = 200          # the assignment requires 200 evaluation games
EVAL_BASE_SEED = 10_000   # evaluation game i is reset with seed EVAL_BASE_SEED + i
FRAMESKIP = 4             # the agent picks an action every 4 frames; the env repeats it 4 times
STICKY_ACTIONS = 0.0      # 0 = deterministic emulator (ALE v5 defaults to 0.25 = random action repeats)

# Raw GitHub URL that fetch_weights() downloads from when a model file is missing locally.
WEIGHTS_BASE_URL = "https://raw.githubusercontent.com/sowmya-subra/rl-opt-project/main/models"

for d in (RESULTS_DIR, MODELS_DIR, CHECKPOINT_DIR, NOTES_DIR):
    os.makedirs(d, exist_ok=True)

## Seeding
`set_global_seeds` makes training runs repeatable and, more importantly, makes the greedy evaluation give **exactly** the same scores on every re-run. On a GPU, tiny floating-point differences can flip which action has the highest Q-value, so we also turn on TensorFlow's deterministic-ops mode.

In [ ]:
def set_global_seeds(seed):
    """Seed Python, NumPy and TensorFlow, and ask TensorFlow for deterministic ops."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.keras.utils.set_random_seed(seed)          # seeds Python `random`, NumPy and TensorFlow together
    tf.config.experimental.enable_op_determinism()  # same inputs -> bit-identical outputs, also on GPU


def hardware_string():
    """Short description of the machine, recorded in <agent>_meta.json for the cost comparison."""
    cpu = platform.processor() or platform.machine()
    gpus = tf.config.list_physical_devices("GPU")
    gpu = ", ".join(tf.config.experimental.get_device_details(g).get("device_name", g.name)
                    for g in gpus) if gpus else "none"
    return f"{platform.system()} {cpu} | GPU: {gpu}"

## Environments
`frameskip=4` is how we "choose an action every 4 frames and play it 4 frames in a row". Sticky actions are off so a given seed always plays out the same way.

In [ ]:
def make_env(game, seed=None, frameskip=FRAMESKIP, sticky=STICKY_ACTIONS, render_mode=None):
    """
    Create an Atari env, e.g. make_env("Pong") or make_env("Breakout").

    frameskip   : frames each chosen action is repeated for (4 = act every 4 frames).
                  Use frameskip=1 to handle frame skipping yourself.
    sticky      : probability the emulator repeats the previous action (0 = off).
    seed        : if given, reset once with it so the env's RNG is fixed.
    render_mode : "rgb_array" if you want to record video, else None.
    """
    env = gym.make(f"ALE/{game}-v5", frameskip=frameskip,
                   repeat_action_probability=sticky, render_mode=render_mode)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env

## Preprocessing: Pong
Pong frames are 210×160 RGB. We crop off the scoreboard, halve the resolution, and turn everything into 0 (background) or 1 (paddles and ball). Pong actions: 0 NOOP, 1 FIRE, 2 RIGHT (= up), 3 LEFT (= down), 4 RIGHTFIRE, 5 LEFTFIRE.

In [ ]:
def preprocess_pong(frame):
    """
    210x160x3 RGB Pong frame -> 80x80 float32 array of 0/1.
    Crops the scoreboard and bottom border, downsamples by 2,
    erases the background, and sets paddles + ball to 1.
    """
    f = frame[35:195]            # crop: drop scoreboard (top) and border (bottom) -> 160x160
    f = f[::2, ::2, 0].copy()    # keep every 2nd pixel and only the red channel -> 80x80
    f[f == 144] = 0              # erase background (shade 1)
    f[f == 109] = 0              # erase background (shade 2)
    f[f != 0] = 1                # paddles and ball -> 1
    return f.astype(np.float32)


def ball_visible_pong(processed):
    """
    True if the ball is on screen. Use it to skip frames with no ball
    (e.g. right after a point), which carry nothing to learn from.
    Ignores the two paddle columns (8-9 = opponent, 70-71 = us). The rare frame
    where the ball sits exactly in a paddle column reads as "no ball".
    """
    return np.delete(processed, [8, 9, 70, 71], axis=1).any()


class PongFrameDiff:
    """
    Turns raw Pong frames into 'current minus previous' images, so one 80x80
    input shows which way the ball is moving (the approach from class).

        fd = PongFrameDiff()
        fd.reset()
        x = fd(obs)   # 80x80 float32, values in {-1, 0, 1}
    """
    def __init__(self):
        self.prev = None

    def reset(self):
        self.prev = None

    def __call__(self, frame):
        cur = preprocess_pong(frame)
        diff = cur - self.prev if self.prev is not None else np.zeros_like(cur)
        self.prev = cur
        return diff

## Preprocessing: Breakout, and frame stacking for any game
Breakout actions: 0 NOOP, 1 FIRE, 2 RIGHT, 3 LEFT. The ball must be launched with FIRE after a reset and after every lost life.

In [ ]:
def preprocess_breakout(frame):
    """
    210x160x3 RGB Breakout frame -> 82x72 float32 grayscale in [0, 1].
    Crops the score bar and side walls, downsamples by 2.
    """
    gray = frame.mean(axis=2)            # RGB -> grayscale
    f = gray[32:196, 8:152][::2, ::2]    # crop to the play area, keep every 2nd pixel
    return (f / 255.0).astype(np.float32)


class FrameStack:
    """
    Keeps the last k preprocessed frames, stacked as (H, W, k), so the
    network can see motion and direction. Works for any game.
    Channels-last (H, W, k) is the layout Keras Conv2D expects, same as the
    class notebooks. The last channel is the newest frame.

        fs = FrameStack(preprocess_pong, k=4)
        x = fs.reset(first_obs)   # (80, 80, 4), first frame repeated 4 times
        x = fs(obs)               # (80, 80, 4), oldest frame dropped
    """
    def __init__(self, preprocess_fn, k=4):
        self.preprocess_fn = preprocess_fn
        self.k = k
        self.frames = []

    def reset(self, obs=None):
        self.frames = []
        if obs is not None:
            return self(obs)

    def __call__(self, obs):
        f = self.preprocess_fn(obs)
        if not self.frames:
            self.frames = [f] * self.k     # start of a game: pad with copies of the first frame
        else:
            self.frames = self.frames[1:] + [f]
        return np.stack(self.frames, axis=-1)


def breakout_fire_needed(info, prev_lives):
    """
    True if Breakout just started or a life was just lost, i.e. the agent must
    press FIRE (action 1) to launch the ball. Inside your episode loop:

        if breakout_fire_needed(info, prev_lives):
            obs, *_ = env.step(1)
        prev_lives = info["lives"]
    """
    return prev_lives is None or info.get("lives", prev_lives) < prev_lives

## Training log
One row per training game. With `resume=True`, the logger reloads the existing CSV after a Colab disconnect, so the game count and wall-clock time pick up where they stopped. That matters for proving Pong-5k's 5,000-game cap and for reporting training cost.

In [ ]:
class TrainingLogger:
    """
    Records one row per training game and writes results/<agent>_train_log.csv.

        logger = TrainingLogger("pong_full", resume=True)
        logger.log(score, frames)          # once at the end of every game
        logger.games_played()              # e.g. to resume a training loop
        logger.save_meta(hyperparams, seed=SEED)

    resume=True  : continue an existing log (game numbers and wall-clock seconds carry on).
    resume=False : start a fresh log (overwrites the old CSV on the next save).
    The CSV is written every `save_every` games; call logger.save() right before saving
    a checkpoint so the log and the weights always match.
    """
    def __init__(self, agent_name, resume=True, save_every=10, window=100):
        self.agent = agent_name
        self.save_every = save_every
        self.window = window
        self.path = os.path.join(RESULTS_DIR, f"{agent_name}_train_log.csv")
        self.rows = []
        prior_seconds = 0.0
        if resume and os.path.exists(self.path):
            self.rows = pd.read_csv(self.path).to_dict("records")
            if self.rows:
                prior_seconds = float(self.rows[-1]["wall_seconds"])
            print(f"TrainingLogger: resuming {agent_name} at game {len(self.rows)}")
        # Shift the clock back so wall_seconds continues from the previous session.
        self.t0 = time.time() - prior_seconds

    def log(self, score, frames=None, **extra):
        """Record one finished game. Extra keyword args (e.g. epsilon=0.1) become extra columns."""
        game = len(self.rows) + 1
        recent = [r["score"] for r in self.rows[-(self.window - 1):]] + [score]
        row = {
            "game": game,
            "score": float(score),
            "running_mean_100": float(np.mean(recent)),
            "frames": frames,
            "wall_seconds": round(time.time() - self.t0, 1),
            **extra,
        }
        self.rows.append(row)
        if game % self.save_every == 0:
            self.save()
        return row

    def save(self):
        pd.DataFrame(self.rows).to_csv(self.path, index=False)

    def games_played(self):
        return len(self.rows)

    def truncate(self, n_games):
        """Drop rows after game n_games, e.g. when resuming from a checkpoint older than the log."""
        self.rows = self.rows[:n_games]
        if self.rows:
            self.t0 = time.time() - float(self.rows[-1]["wall_seconds"])
        self.save()

    def save_meta(self, hyperparams, seed=None, notes=""):
        """Write results/<agent>_meta.json with hyperparameters, training time and hardware."""
        meta = {
            "agent": self.agent,
            "training_games": len(self.rows),
            "training_hours": round((time.time() - self.t0) / 3600, 3),
            "seed": seed,
            "hyperparameters": hyperparams,
            "hardware": hardware_string(),
            "notes": notes,
        }
        with open(os.path.join(RESULTS_DIR, f"{self.agent}_meta.json"), "w") as fh:
            json.dump(meta, fh, indent=2, default=str)
        self.save()
        return meta

## Loading saved weights
The TA only receives the `.ipynb` and HTML, not our `models/` folder. When a notebook runs with `TRAIN = False` and the weights file is missing, `fetch_weights` downloads it from our GitHub repo first.

In [ ]:
def fetch_weights(agent_name, ext="weights.h5", base_url=WEIGHTS_BASE_URL):
    """
    Return the local path to models/<agent_name>.<ext>, downloading it from
    base_url if it isn't on disk yet. Load it into a freshly built model, e.g.
        model.load_weights(fetch_weights("pong_full"))
    """
    path = os.path.join(MODELS_DIR, f"{agent_name}.{ext}")
    if not os.path.exists(path):
        url = f"{base_url}/{agent_name}.{ext}"
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, path)
    return path

## Evaluation
`evaluate` produces the numbers we report and the TA re-runs. Game *i* is always reset with seed `EVAL_BASE_SEED + i`, so the same 200 games are played every time. Score = sum of rewards in a game: for Pong that's (our points − opponent points), so a mean above 0 is a winning score.

In [ ]:
def evaluate(env_fn, policy, n_games=EVAL_GAMES, agent_name=None,
             base_seed=EVAL_BASE_SEED, max_steps=50_000, verbose=True):
    """
    Play n_games full games with a trained policy (no learning, no exploration)
    and return a NumPy array of per-game scores.

    env_fn     : function returning a fresh env, e.g. lambda: make_env("Pong")
    policy     : object with .act(obs) -> action and optional .reset()
                 (a plain function obs -> action also works). Must be greedy.
    agent_name : if given, writes results/<agent_name>_eval_<n_games>.csv
    base_seed  : game i is reset with seed base_seed + i
    max_steps  : safety cap on agent steps per game, so a stalled game still ends
    """
    act = policy.act if hasattr(policy, "act") else policy
    env = env_fn()
    scores, lengths = [], []
    for i in range(n_games):
        obs, info = env.reset(seed=base_seed + i)
        env.action_space.seed(base_seed + i)
        if hasattr(policy, "reset"):
            policy.reset()
        total, steps, done = 0.0, 0, False
        while not done and steps < max_steps:
            obs, r, terminated, truncated, info = env.step(act(obs))
            total += r
            steps += 1
            done = terminated or truncated
        scores.append(total)
        lengths.append(steps)
        if verbose and (i + 1) % 20 == 0:
            print(f"  eval {i + 1}/{n_games}: running mean = {np.mean(scores):.2f}")
    env.close()
    scores = np.array(scores)
    if agent_name:
        pd.DataFrame({"game": np.arange(1, n_games + 1), "score": scores,
                      "steps": lengths}).to_csv(
            os.path.join(RESULTS_DIR, f"{agent_name}_eval_{n_games}.csv"), index=False)
    if verbose:
        print(f"{agent_name or 'policy'}: mean {scores.mean():.2f} +/- {scores.std():.2f} "
              f"over {n_games} games (games with score > 0: {(scores > 0).sum()})")
    return scores


class RandomPolicy:
    """Uniform random actions: the 'no skill' baseline. Seeded, so it is reproducible."""
    def __init__(self, n_actions, seed=0):
        self.n = n_actions
        self.rng = np.random.default_rng(seed)

    def act(self, obs):
        return int(self.rng.integers(self.n))


def random_baseline(game, n_games=EVAL_GAMES):
    """Evaluate a random agent and save results/<game>_random_eval_<n_games>.csv."""
    n_actions = make_env(game).action_space.n
    return evaluate(lambda: make_env(game), RandomPolicy(n_actions),
                    n_games=n_games, agent_name=f"{game.lower()}_random")

## Smoke test
Set `RUN_SMOKE_TEST = True` once to check that your install works. It plays 3 random Pong games twice and confirms both runs give the same scores (it writes a throwaway `results/pong_random_eval_3.csv`).

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    set_global_seeds(0)
    print("hardware:", hardware_string())
    for g in ["Pong", "Breakout"]:
        env = make_env(g, seed=0)
        obs, _ = env.reset(seed=0)
        print(g, "obs", obs.shape, "actions", env.unwrapped.get_action_meanings())
    pong_obs, _ = make_env("Pong").reset(seed=0)
    print("pong processed:", preprocess_pong(pong_obs).shape,
          "| pong stacked:", FrameStack(preprocess_pong).reset(pong_obs).shape,
          "| breakout processed:", preprocess_breakout(obs).shape)
    a = random_baseline("Pong", n_games=3)
    b = random_baseline("Pong", n_games=3)
    print("reproducible:", np.array_equal(a, b))

---
# Your agent starts here
Everything above this line is shared. Leave it unchanged in your copy. Put your agent's code below.

# Part 1: Pong agent trained until it wins (`pong_full`)

**Goal:** an agent whose average score over 200 evaluation games is above 0 (it scores more points than the computer opponent).

**Starting point: the class engines.** The Q-ish engine from Module 10 (`dqnish_500_train_20_eval.ipynb`) trained for 500 games and then scored **−21.0** in all 20 evaluation games: it never returned a single ball. That is our "before" number.

**What we changed, and why.** We kept the class network and preprocessing and fixed how it learns:

| Class engine | This agent | Why it matters |
| --- | --- | --- |
| Trains once per game on that game's frames only | **Replay buffer**: stores 50,000 recent moves; trains on random mini-batches of 32 after every move | Consecutive frames are nearly identical. Random batches break that correlation, and each experience is reused many times |
| Target = discounted final reward of the point (+1/−1) | **Bellman target with a target network**: `r + 0.99 · Q_target(s', best a')` | The agent learns the value of a position from the next position, so credit flows back through the rally; a frozen copy of the network (synced every 1,000 moves) keeps the target from chasing itself |
| ε from 0.95 → 0.05 by game | **Linearly annealed ε** from 1.0 → 0.02 over the first 150,000 moves, then fixed | Lots of exploration early, almost pure exploitation once it knows what it's doing |
| MSE loss | Huber loss, gradient clipping, **Double DQN** | Standard stabilisers: one bad batch can't blow up the weights, and Q-values aren't systematically over-estimated |

Kept from the assignment's list: an action every 4 frames (`FRAMESKIP = 4`), 4 stacked 80×80 frames, the 3 useful buttons (NOOP, UP, DOWN), the memory buffer, linear annealing.
Not used: reward shaping and skipping no-ball frames, because the Bellman target already rewards a good return indirectly (it leads to won points), and extra shaped rewards risk teaching the agent to prolong rallies.

## Settings
`TRAIN = False` is what the grader runs: it downloads our trained weights (if needed) and only evaluates. Set `TRAIN = True` to train; training resumes automatically from the latest checkpoint after a disconnect.

**On Colab**, mount Google Drive and `cd` into the project folder first so checkpoints survive a disconnect:
```python
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/rl-opt-project')
```

In [ ]:
import pickle
import shutil
import keras
from keras.layers import Input, Conv2D, Flatten, Dense

AGENT = "pong_full"      # file names: results/pong_full_*.csv, models/pong_full.weights.h5
TRAIN = False            # False = load trained weights and evaluate only (what the grader runs)
SEED = 294               # one seed for TF, NumPy and Python (set in set_global_seeds)

PONG_ACTIONS = [0, 2, 3]       # network output i -> env button: 0 NOOP, 2 RIGHT (= up), 3 LEFT (= down)
N_ACTIONS = len(PONG_ACTIONS)
STATE_SHAPE = (80, 80, 4)      # 4 stacked 80x80 frames, channels-last

HP = dict(
    gamma=0.99,              # discount factor: how much a point 100 moves from now counts today
    learning_rate=1e-4,      # Adam step size (same as the class models)
    batch_size=32,           # moves per gradient step, sampled at random from the buffer
    buffer_size=50_000,      # replay buffer capacity (moves)
    replay_start=10_000,     # fill the buffer with this many moves before training starts
    train_every=1,           # gradient steps happen every `train_every` moves
    target_sync=1_000,       # copy the online net into the target net every 1,000 moves
    eps_start=1.0,           # epsilon (chance of a random move) at the start ...
    eps_end=0.02,            # ... decays linearly to this ...
    eps_decay_steps=150_000, # ... over this many moves, then stays flat
    double_dqn=True,         # pick a' with the online net, value it with the target net
    grad_clip=10.0,          # max global gradient norm
    checkpoint_every=100,    # games between checkpoints
    stop_running_mean=18.0,  # stop once the 100-game training average reaches this ...
    max_games=2_000,         # ... or after this many games, whichever comes first
)

## The Q-network
Same architecture as the class models (the 2013 DeepMind Atari network): two convolution layers read the 4 stacked frames, a dense layer combines them, and the output layer gives one Q-value per button, i.e. the expected future score of pressing it.

In [ ]:
def create_q_model(learning_rate=HP["learning_rate"], for_training=True):
    """
    Class Q-network: 80x80x4 stacked frames in, one Q-value per button out.
    for_training=False builds it without an optimizer (evaluation only needs the weights).
    """
    inp = Input(shape=STATE_SHAPE)
    x = Conv2D(16, (8, 8), strides=4, activation="relu")(inp)   # -> 19x19x16
    x = Conv2D(32, (4, 4), strides=2, activation="relu")(x)     # -> 8x8x32
    x = Flatten()(x)
    x = Dense(256, activation="relu")(x)
    out = Dense(N_ACTIONS, activation="linear")(x)              # Q-values, not probabilities
    model = keras.Model(inp, out)
    # Compiling attaches the optimizer, so model.save() also stores Adam's state for resuming.
    if for_training:
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate))
    return model


def make_q_fn(model):
    """Fast compiled forward pass used for choosing moves (fixed input shape, so no retracing)."""
    @tf.function(input_signature=[tf.TensorSpec((None,) + STATE_SHAPE, tf.float32)])
    def q_fn(x):
        return model(x, training=False)
    return q_fn


create_q_model().summary()

## Replay buffer
Stores `(state, action, reward, next_state, done)` for the most recent moves. Pong frames are only 0s and 1s, so each 80×80×4 state is **bit-packed** into 3,200 bytes (8× smaller than uint8): 50,000 moves take ~320 MB instead of ~2.6 GB.

In [ ]:
class ReplayBuffer:
    """Circular buffer of moves; the oldest move is overwritten once it is full."""

    def __init__(self, capacity, state_shape=STATE_SHAPE):
        self.capacity = capacity
        self.state_shape = state_shape
        packed = int(np.prod(state_shape)) // 8            # bytes per bit-packed state
        self.s = np.zeros((capacity, packed), dtype=np.uint8)
        self.s2 = np.zeros((capacity, packed), dtype=np.uint8)
        self.a = np.zeros(capacity, dtype=np.int32)
        self.r = np.zeros(capacity, dtype=np.float32)
        self.done = np.zeros(capacity, dtype=np.float32)
        self.ptr = 0    # where the next move is written
        self.n = 0      # how many moves are stored

    def add(self, s, a, r, s2, done):
        i = self.ptr
        self.s[i] = np.packbits(s.astype(np.uint8))     # 0/1 floats -> bits
        self.s2[i] = np.packbits(s2.astype(np.uint8))
        self.a[i], self.r[i], self.done[i] = a, r, float(done)
        self.ptr = (self.ptr + 1) % self.capacity
        self.n = min(self.n + 1, self.capacity)

    def _unpack(self, packed):
        return np.unpackbits(packed, axis=1).reshape((-1,) + self.state_shape).astype(np.float32)

    def sample(self, batch_size, rng):
        """Random mini-batch as float32 arrays, ready for the network."""
        idx = rng.integers(0, self.n, size=batch_size)
        return (self._unpack(self.s[idx]), self.a[idx], self.r[idx],
                self._unpack(self.s2[idx]), self.done[idx])

    def save(self, path):
        np.savez(path, s=self.s[:self.n], s2=self.s2[:self.n], a=self.a[:self.n],
                 r=self.r[:self.n], done=self.done[:self.n], ptr=self.ptr)

    def load(self, path):
        d = np.load(path)
        self.n = len(d["a"])
        self.s[:self.n], self.s2[:self.n] = d["s"], d["s2"]
        self.a[:self.n], self.r[:self.n], self.done[:self.n] = d["a"], d["r"], d["done"]
        self.ptr = int(d["ptr"])

## Learning step and exploration schedule
One gradient step of DQN. The target for the move we took is the reward plus the discounted value of the next state:

$$y = r + \gamma \,(1-\text{done})\; Q_{\text{target}}\big(s',\ \arg\max_{a'} Q_{\text{online}}(s', a')\big)$$

and we nudge $Q_{\text{online}}(s, a)$ toward $y$ with a Huber loss. Choosing $a'$ with the online net but valuing it with the target net is **Double DQN**.

In [ ]:
def make_train_step(q_net, target_net, gamma=HP["gamma"],
                    double_dqn=HP["double_dqn"], grad_clip=HP["grad_clip"]):
    """Build a compiled function that does one DQN gradient step on a mini-batch."""
    huber = keras.losses.Huber()
    optimizer = q_net.optimizer

    @tf.function
    def train_step(s, a, r, s2, done):
        # --- target y (no gradient flows through this part) ---
        q_next_target = target_net(s2, training=False)
        if double_dqn:
            best_next = tf.argmax(q_net(s2, training=False), axis=1)   # online net picks a'
        else:
            best_next = tf.argmax(q_next_target, axis=1)
        q_next = tf.reduce_sum(q_next_target * tf.one_hot(best_next, N_ACTIONS), axis=1)
        y = r + gamma * (1.0 - done) * q_next                          # done -> no future value

        # --- move Q(s, a) toward y ---
        with tf.GradientTape() as tape:
            q = q_net(s, training=True)
            q_taken = tf.reduce_sum(q * tf.one_hot(a, N_ACTIONS), axis=1)  # Q of the move we made
            loss = huber(y, q_taken)
        grads = tape.gradient(loss, q_net.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, grad_clip)
        optimizer.apply_gradients(zip(grads, q_net.trainable_variables))
        return loss

    return train_step


def epsilon_at(step, hp=HP):
    """Linear annealing: eps_start -> eps_end over eps_decay_steps moves, then flat."""
    frac = min(1.0, step / hp["eps_decay_steps"])
    return hp["eps_start"] + frac * (hp["eps_end"] - hp["eps_start"])

## Checkpoints (resume after a Colab disconnect)
Every 100 games we save everything needed to continue exactly where we left off: the online network with its Adam state, the target network, the replay buffer, the move counter and the random-number state. Files are written to a temporary folder first and then swapped in, so a disconnect in the middle of saving never leaves a half-written checkpoint. A copy of the weights at each checkpoint is also kept (`pong_full_game<N>.weights.h5`) to show how the agent improved.

In [ ]:
LATEST_DIR = os.path.join(CHECKPOINT_DIR, f"{AGENT}_latest")


def save_checkpoint(game, step, q_net, target_net, buffer, rng, logger):
    tmp = LATEST_DIR + "_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    q_net.save(os.path.join(tmp, "q_net.keras"))                     # weights + Adam state
    target_net.save_weights(os.path.join(tmp, "target.weights.h5"))
    buffer.save(os.path.join(tmp, "buffer.npz"))
    with open(os.path.join(tmp, "state.pkl"), "wb") as fh:
        pickle.dump({"game": game, "step": step, "rng": rng.bit_generator.state}, fh)
    logger.save()
    shutil.rmtree(LATEST_DIR, ignore_errors=True)                    # swap the new checkpoint in
    os.replace(tmp, LATEST_DIR)
    q_net.save_weights(os.path.join(CHECKPOINT_DIR, f"{AGENT}_game{game}.weights.h5"))
    print(f"  checkpoint saved at game {game} ({step:,} moves)")


def load_checkpoint():
    """Return (game, step, q_net, target_net, buffer, rng) from the latest checkpoint, or None."""
    if not os.path.exists(os.path.join(LATEST_DIR, "state.pkl")):
        return None
    with open(os.path.join(LATEST_DIR, "state.pkl"), "rb") as fh:
        state = pickle.load(fh)
    q_net = keras.models.load_model(os.path.join(LATEST_DIR, "q_net.keras"))
    target_net = create_q_model(for_training=False)   # the target net is never trained directly
    target_net.load_weights(os.path.join(LATEST_DIR, "target.weights.h5"))
    buffer = ReplayBuffer(HP["buffer_size"])
    buffer.load(os.path.join(LATEST_DIR, "buffer.npz"))
    rng = np.random.default_rng()
    rng.bit_generator.state = state["rng"]
    print(f"Resuming from checkpoint: game {state['game']}, {state['step']:,} moves")
    return state["game"], state["step"], q_net, target_net, buffer, rng

## Training loop
One call plays training games until the 100-game running mean reaches `stop_running_mean` (or `max_games`). Each move: choose an action (random with probability ε, otherwise the highest Q-value), play it for 4 frames, store the move in the buffer, and take one gradient step on a random mini-batch.

In [ ]:
TRAIN_SEED_OFFSET = 1_000_000   # training game g uses env seed 1,000,000 + g (never overlaps the eval seeds)


def train_agent(hp=HP, agent=AGENT):
    set_global_seeds(SEED)
    logger = TrainingLogger(agent, resume=True)

    # Fresh start, or pick up from the latest checkpoint
    ckpt = load_checkpoint()
    if ckpt is None:
        game, step = 0, 0
        q_net, target_net = create_q_model(), create_q_model(for_training=False)
        target_net.set_weights(q_net.get_weights())
        buffer = ReplayBuffer(hp["buffer_size"])
        rng = np.random.default_rng(SEED)
        logger.truncate(0)
    else:
        game, step, q_net, target_net, buffer, rng = ckpt
        logger.truncate(game)       # drop log rows from games played after the checkpoint

    q_fn = make_q_fn(q_net)
    train_step = make_train_step(q_net, target_net, hp["gamma"], hp["double_dqn"], hp["grad_clip"])
    env = make_env("Pong")
    frames = FrameStack(preprocess_pong, k=4)

    while game < hp["max_games"]:
        game += 1
        obs, _ = env.reset(seed=TRAIN_SEED_OFFSET + game)
        state = frames.reset(obs)
        score, moves, losses, done = 0.0, 0, [], False
        t_game = time.time()

        while not done:
            # 1. choose an action: explore with probability epsilon, otherwise act greedily
            eps = epsilon_at(step, hp)
            if rng.random() < eps:
                a = int(rng.integers(N_ACTIONS))
            else:
                a = int(np.argmax(q_fn(state[None])[0]))

            # 2. play it (the env repeats it for 4 frames) and build the next stacked state
            obs, r, terminated, truncated, _ = env.step(PONG_ACTIONS[a])
            next_state = frames(obs)
            done = terminated or truncated

            # 3. remember the move. Only a real game end (terminated) zeroes the future value
            buffer.add(state, a, r, next_state, terminated)
            state = next_state
            score += r
            step += 1
            moves += 1

            # 4. learn from a random mini-batch once the buffer has enough moves
            if buffer.n >= hp["replay_start"] and step % hp["train_every"] == 0:
                losses.append(float(train_step(*buffer.sample(hp["batch_size"], rng))))

            # 5. periodically refresh the frozen target network
            if step % hp["target_sync"] == 0:
                target_net.set_weights(q_net.get_weights())

        row = logger.log(score, frames=moves * FRAMESKIP, moves=moves, epsilon=round(eps, 4),
                         loss=round(float(np.mean(losses)), 5) if losses else None,
                         buffer=buffer.n, total_moves=step)
        if game % 10 == 0 or game == 1:
            print(f"game {game:5d} | score {score:+4.0f} | mean100 {row['running_mean_100']:+6.2f} "
                  f"| eps {eps:.3f} | loss {row['loss']} | moves {step:,} "
                  f"| {time.time() - t_game:5.1f}s/game | {row['wall_seconds'] / 3600:.2f} h total")

        if game % hp["checkpoint_every"] == 0:
            save_checkpoint(game, step, q_net, target_net, buffer, rng, logger)

        if game >= 100 and row["running_mean_100"] >= hp["stop_running_mean"]:
            print(f"Reached a 100-game mean of {row['running_mean_100']:.2f} at game {game}. Stopping.")
            break

    env.close()
    if game % hp["checkpoint_every"] != 0:          # not already saved this game
        save_checkpoint(game, step, q_net, target_net, buffer, rng, logger)
    q_net.save_weights(os.path.join(MODELS_DIR, f"{agent}.weights.h5"))   # the final model
    meta = logger.save_meta({**hp, "actions": PONG_ACTIONS, "state_shape": STATE_SHAPE,
                             "frameskip": FRAMESKIP, "sticky_actions": STICKY_ACTIONS,
                             "total_moves": step},
                            seed=SEED, notes="DQN: replay buffer + target network + Double DQN")
    print(f"Done: {meta['training_games']} games, {meta['training_hours']} h on {meta['hardware']}")
    return q_net

In [ ]:
if TRAIN:
    train_agent()

## Learning curve
Score of every training game and the 100-game running mean. The dashed line at 0 is the break-even score: above it, the agent wins more points than it loses.

In [ ]:
import matplotlib.pyplot as plt

log_path = os.path.join(RESULTS_DIR, f"{AGENT}_train_log.csv")
if os.path.exists(log_path):
    log = pd.read_csv(log_path)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(log["game"], log["score"], color="#9aa5b1", linewidth=0.8, label="score per game")
    ax.plot(log["game"], log["running_mean_100"], color="#1f5fa8", linewidth=2, label="100-game running mean")
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(xlabel="training game", ylabel="score (our points - opponent points)",
           ylim=(-21.5, 21.5), title="Pong DQN: training progress")
    ax.legend(loc="lower right")
    plt.show()
else:
    print(f"No training log at {log_path} yet.")

## Evaluation: 200 games, greedy, no training
This is the number we report. The trained weights are loaded from `models/pong_full.weights.h5` (downloaded from GitHub if missing). The agent always presses the button with the highest Q-value: no exploration and no learning. Game *i* uses seed `10,000 + i`, and we play all 200 games twice to show the result reproduces exactly.

In [ ]:
class GreedyPongPolicy:
    """Trained agent for evaluation: raw frame in, env button out, always the best Q-value."""

    def __init__(self, model):
        self.q_fn = make_q_fn(model)
        self.frames = FrameStack(preprocess_pong, k=4)

    def reset(self):
        self.frames.reset()            # new game: forget the previous game's frames

    def act(self, obs):
        state = self.frames(obs)       # preprocess + stack, same as in training
        q = self.q_fn(state[None])[0]
        return PONG_ACTIONS[int(np.argmax(q))]


set_global_seeds(SEED)
eval_model = create_q_model(for_training=False)
eval_model.load_weights(fetch_weights(AGENT))

scores_run1 = evaluate(lambda: make_env("Pong"), GreedyPongPolicy(eval_model), agent_name=AGENT)

In [ ]:
# Re-run the identical 200 games to prove the reported score reproduces exactly.
scores_run2 = evaluate(lambda: make_env("Pong"), GreedyPongPolicy(eval_model), verbose=False)
print("Identical scores on re-run:", np.array_equal(scores_run1, scores_run2))
print(f"Reported 200-game mean: {scores_run1.mean():.2f}  "
      f"(won {(scores_run1 > 0).sum()} of {len(scores_run1)} games)")